In [1]:
import pandas as pd
import tarfile
import xml.etree.ElementTree as ET
from xml.dom.minidom import parse, parseString
from collections import Counter
import numpy as np
import re
import csv
from tqdm import tqdm
import csv
import pickle
import matplotlib.pyplot as plt
import time
import os
import lxml
from lxml import etree
import xmltodict
from collections.abc import MutableMapping
import copy

In [14]:
def tar_extract_and_populate(location_input: str):
    """This function takes in a set of tar files, extracts them in a directory of choice and populates a list of files in the directory"""
    
    tar_data_files = [name for name in os.listdir(location_input)]
    xml_files_list=[]
    year = location_input[-4:]
    
    for tar_data_file in tar_data_files:
        #open the tar.gz file as a tar archive
        print("{}/{}".format(location_input, tar_data_file))
        tar_archive = tarfile.open(name="{}/{}".format(location_input, tar_data_file), mode="r:gz")

        #extract the tar archive in a directory of choice
        tar_archive.extractall("new_data/extracted_data/")

        for file in tar_archive.getnames():
            if file[-4:] == ".xml":
                xml_files_list.append("{}/{}".format(location_input, file))

        tar_archive.close()

    return xml_files_list

In [15]:
directory = "new_data/raw_data"
xml_files_list = tar_extract_and_populate(directory)

new_data/raw_data/2020-01.tar.gz
new_data/raw_data/2020-02.tar.gz
new_data/raw_data/2020-03.tar.gz
new_data/raw_data/2020-04.tar.gz
new_data/raw_data/2020-05.tar.gz
new_data/raw_data/2020-06.tar.gz
new_data/raw_data/2020-07.tar.gz
new_data/raw_data/2020-08.tar.gz
new_data/raw_data/2020-09.tar.gz
new_data/raw_data/2020-10.tar.gz
new_data/raw_data/2020-11.tar.gz
new_data/raw_data/2020-12.tar.gz


In [27]:
new_xml_file_list = []
for file in xml_files_list:
    new_xml_file_list.append(file.replace("raw_data", "extracted_data"))

In [34]:
xml_files_list = new_xml_file_list

In [29]:
#with open("new_data/files_list.pickle", "wb") as f:
#    pickle.dump(new_xml_file_list, f)

In [19]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [20]:
def flatten_dict(d, parent_key ='', sep ='_'):
    items = []
    for k, v in d.items():
        new_key = parent_key + sep + k if parent_key else k
 
        if isinstance(v, MutableMapping):
            items.extend(flatten_dict(v, new_key, sep = sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

In [35]:
data_dict = {}

for i, file_path in tqdm(enumerate(xml_files_list)):

    with open(file_path, "r", encoding="utf-8", errors="ignore") as fileptr:
        xml_content = fileptr.read()

    tender_dict = xmltodict.parse(xml_content, process_namespaces=False)
    tender_flat_dict = flatten_dict(tender_dict)
    data_dict[i] = tender_flat_dict

195257it [1:22:23, 76.55it/s] 

observations on key-value pairs:<br>
TED_EXPORT: <br>
@xmlns --> xmlns value <br>
@DOC_ID --> primary key <br>
NO_OJ --> number of objects? does not seem like it<br>

NOTICE_DATA: <br>
LG_ORIG --> original language <br>
ISO_COUNTRY --> country code <br>

Workflow:
for each document
    if number of lots > 1:
        make row per lot
    else:
        make row per document
    
    subroutine for getting row per lot: <br>
    get document id
        


In [7]:
#get the row and short description out of the dict
extracted_data = []
base_string = r'TED_EXPORT_FORM_SECTION_F\d{2}_2014_OBJECT_CONTRACT_OBJECT_DESCR'
short_descr_pattern = r'TED_EXPORT_FORM_SECTION_F\d{2}_2014_OBJECT_CONTRACT_SHORT_DESCR_P(?!_.*)'
short_descr_pattern_lots = base_string + r'(?!_.*)'
duration_type_pattern = base_string + r'_DURATION_@TYPE'
duration_text_pattern = base_string + r'_DURATION_#text'
title_text_pattern = r'TED_EXPORT_FORM_SECTION_F03_2014_OBJECT_CONTRACT_TITLE_P'
doc_id_pattern = r'0*([1-9][0-9]*|0[1-9][0-9]*)'

#TED_EXPORT_FORM_SECTION_F03_2014_OBJECT_CONTRACT_SHORT_DESCR_P
count = 0
count_nested_dicts_in_list = 0
for i, file_number in tqdm(enumerate(data_dict.keys())):
    tender = {"ID_NOTICE_CAN": "",
              "LG_ORIG": "",
              "short description": "",
              "duration type": "",
              "duration": "",
              "title": ""}
    
    for tag in data_dict[file_number].keys():
        if "@DOC_ID" in tag:
            doc_id_list = str(data_dict[file_number][tag]).split("-")
            doc_id_number = re.findall(doc_id_pattern, doc_id_list[0])
            doc_id = doc_id_list[1] + doc_id_number[0]
            tender["ID_NOTICE_CAN"] = doc_id
        
        elif "LG_ORIG" in tag:
            if type(data_dict[file_number][tag]) == list:
                tender["LG_ORIG"] = data_dict[file_number][tag][0]
            else:
                tender["LG_ORIG"] = data_dict[file_number][tag]
        
        elif re.match(short_descr_pattern, tag):
            if type(data_dict[file_number][tag]) == list:
                if type(data_dict[file_number][tag][0]) == str:
                    tender["short description"] = data_dict[file_number][tag][0]
                else:
                    count_nested_dicts_in_list += 1
            else:
                tender["short description"] = data_dict[file_number][tag]
        elif re.match(duration_type_pattern, tag):
            tender["duration type"] = data_dict[file_number][tag]
        elif re.match(duration_text_pattern, tag):
            tender["duration"] = data_dict[file_number][tag]
        elif re.match(title_text_pattern, tag):
            tender["title"] = data_dict[file_number][tag]
        elif re.match(short_descr_pattern_lots, tag) and type(data_dict[file_number][tag]) == list:
            count += 1
            for lot_number in range(0, len(data_dict[file_number][tag])):
                base_dict_checkpoint = copy.deepcopy(tender)
                flat_lot_notice = flatten_dict(data_dict[file_number][tag][lot_number])
                #print(flat_lot_notice)
                for variable in flat_lot_notice:
                    if variable == "SHORT_DESCR_P":
                        if type(flat_lot_notice["SHORT_DESCR_P"]) == list:
                            if type(flat_lot_notice["SHORT_DESCR_P"][0]) == str:
                                base_dict_checkpoint["short description"] = flat_lot_notice["SHORT_DESCR_P"][0]
                            else:
                                count_nested_dicts_in_list +=1
                        else:
                            base_dict_checkpoint["short description"] = flat_lot_notice["SHORT_DESCR_P"]
                    elif variable == "LOT_NO":
                        base_dict_checkpoint["ID_LOT"] = flat_lot_notice["LOT_NO"]
                    elif "DURATION_@TYPE" in variable:
                        base_dict_checkpoint["duration type"] = flat_lot_notice[variable]
                    elif "DURATION_#text" in variable:
                        base_dict_checkpoint["duration"] = flat_lot_notice[variable]
                    elif "TITLE_P" in variable:
                        base_dict_checkpoint["title"] = flat_lot_notice[variable]
                    else:
                        continue

                #ADD NaN FOR MISSING VALUES
                for variable in base_dict_checkpoint.keys():
                    if len(base_dict_checkpoint[variable]) < 1:
                        base_dict_checkpoint[variable] = np.nan
                    else:
                        continue
                
                extracted_data.append(base_dict_checkpoint)

            tender = {}
        else:
            continue
    
    if len(tender) > 0:
       for variable in tender.keys():
            if len(tender[variable]) < 1:
               tender[variable] = np.nan
            else:
               continue
           
       tender["ID_LOT"] = 0
       extracted_data.append(tender)


1000it [00:00, 1426.74it/s]


In [8]:
extracted_data

[{'ID_NOTICE_CAN': '2020371',
  'LG_ORIG': 'CS',
  'short description': 'Předmětem plnění veřejné zakázky je dodávka pumpy PIPAC pro I. CHK Všeobecné fakultní nemocnice v Praze, včetně příslušenství, cla, dopravy, instalace, uvedení do provozu, vstupní validace, provedení instruktáže dle příslušného ustanovení zákona č. 268/2014 Sb. o zdravotnických prostředcích (dále jen zákona č. 268/2014 Sb.), popř. zaškolení obsluhy kupujícího a předání dokladů, které se k\u202fdodávanému zboží vztahují, zejména prohlášení o shodě a návod k\u202fobsluze v\u202fčeském jazyce v\u202ftištěné i elektronické podobě, včetně záručního servisu a dále dle ostatních podmínek zadávací dokumentace.',
  'duration type': nan,
  'duration': nan,
  'title': 'PIPAC - pumpa + nebulizéry',
  'ID_LOT': 0},
 {'ID_NOTICE_CAN': '2020435',
  'LG_ORIG': 'ES',
  'short description': 'El objeto del presente contrato es el desarrollo de los trabajos relacionados con las funciones básicas de documentar las colecciones museográ

In [9]:
df_csv = pd.read_pickle("new_data/df_from_csv_preprocessed.pickle")

In [10]:
#MAKE DF FROM DICT
df = pd.DataFrame.from_records(extracted_data)

In [11]:
#make list of similar ID_NOTICE_CAN values
id_notice_can_csv_col = list(df_csv["ID_NOTICE_CAN"].values)

shared_values = []
for i in tqdm(range(0, len(df))):
    if int(df["ID_NOTICE_CAN"].iloc[i]) in id_notice_can_csv_col:
        shared_values.append(i)

100%|██████████| 3948/3948 [00:36<00:00, 108.95it/s]


In [12]:
def make_id_lot(df):
    values = [1]
    count = 1
    for i in tqdm(range(1, len(df))):
        if df["ID_NOTICE_CAN"].iloc[i] == df["ID_NOTICE_CAN"].iloc[i-1]:
            count += 1
            values.append(count)
        
        elif df["ID_NOTICE_CAN"].iloc[i] != df["ID_NOTICE_CAN"].iloc[i-1]:
            count = 1
            values.append(count)
    return values

In [18]:
def create_new_index(df):
    new_index = []
    for i in tqdm(range(0, len(df))):
        id_notice_can = df["ID_NOTICE_CAN"].iloc[i]
        id_lot = df["ID_LOT"].iloc[i]
        new_index.append(str(id_notice_can) + "-" + str(id_lot))

    df["pk"] = new_index
    df = df.set_index("pk")
    return df

In [19]:
df["ID_LOT"] = make_id_lot(df)
df_csv["ID_LOT"] = make_id_lot(df_csv)

df = create_new_index(df)
df_csv = create_new_index(df_csv)

100%|██████████| 445308/445308 [00:07<00:00, 59637.50it/s]


In [59]:
def combine_df(df_main, df_supplementary):
    list_indices_main = list(df_main.index.values)
    list_indices_supplementary = list(df_supplementary.index.values)
    shared_list = list(set(list_indices_supplementary).intersection(list_indices_main))
    #retrieve subsection of supplementary dataframe
    print(shared_list)
    df_supplementary_selection = df_supplementary.loc[shared_list]
    df_main_selection          = df_main.loc[shared_list]

    df_main_selection[df_supplementary.columns] = df_supplementary_selection
    return df_main_selection

In [ ]:
df_final = combine_df(df, df_csv)